# RAG-Based Document Assistant

## NLP Project – B.Tech CSE (AI & ML)

### Project Overview
This project uses Retrieval-Augmented Generation (RAG) to answer questions from a university programme handbook.

### Technologies Used
- Python
- NLP
- Sentence Transformers
- FAISS Vector Database
- Ollama (Llama 3.2)
- Streamlit
- PyPDF

In [2]:
# Import Required Libraries

import os
import re
import numpy as np
import pandas as pd
import faiss
import requests

from io import BytesIO
from pathlib import Path
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
# Load University Programme Handbook

pdf_path = Path("Document/handbook.pdf.pdf")

if pdf_path.exists():
    reader = PdfReader(str(pdf_path))

    print("PDF loaded successfully!")
    print("Total Pages:", len(reader.pages))

else:
    print("PDF not found. Please check the file path.")

PDF loaded successfully!
Total Pages: 354


In [4]:
# Extract Text from All PDF Pages

all_pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""

    if text.strip():
        all_pages.append({
            "page": page_number,
            "text": text
        })

print("Text extraction completed!")
print("Total pages extracted:", len(all_pages))
print("\nSample extracted text:\n")
print(all_pages[0]["text"][:1000])

Text extraction completed!
Total pages extracted: 352

Sample extracted text:

 
 
SCHOOL OF ENGINEERING AND TECHNOLOGY 
 
Programme Handbook 
(Programme Study and Evaluation Scheme) 
 
B.Tech (Computer Science & Engineering) with 
Specialization in AI & ML 
Programme Code: 73 
 
FOUR YEAR UNDERGRADUATE PROGRAMME 
As per National Education Policy 2020  
(Multiple Entry and Exit in Academic Programmes)  
(with effect from 2024 -25 session)  
 
Approved in the 34th Meeting of Academic Council Held on 
29 June 2024 
 
 
 
 
 



In [5]:
# Clean Extracted Text

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    return text.strip()


for page in all_pages:
    page["clean_text"] = clean_text(page["text"])

print("Text cleaning completed!")

print("\nOriginal Text:")
print(all_pages[0]["text"][:300])

print("\nCleaned Text:")
print(all_pages[0]["clean_text"][:300])

Text cleaning completed!

Original Text:
 
 
SCHOOL OF ENGINEERING AND TECHNOLOGY 
 
Programme Handbook 
(Programme Study and Evaluation Scheme) 
 
B.Tech (Computer Science & Engineering) with 
Specialization in AI & ML 
Programme Code: 73 
 
FOUR YEAR UNDERGRADUATE PROGRAMME 
As per National Education Policy 2020  
(Multiple Entry and Exi

Cleaned Text:
SCHOOL OF ENGINEERING AND TECHNOLOGY Programme Handbook (Programme Study and Evaluation Scheme) B.Tech (Computer Science & Engineering) with Specialization in AI & ML Programme Code: 73 FOUR YEAR UNDERGRADUATE PROGRAMME As per National Education Policy 2020 (Multiple Entry and Exit in Academic Progr


In [6]:
# Divide Text into Smaller Chunks

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = []

for page in all_pages:
    page_chunks = splitter.split_text(page["clean_text"])

    for chunk in page_chunks:
        chunks.append({
            "page": page["page"],
            "text": chunk
        })

print("Text chunking completed!")
print("Total chunks created:", len(chunks))

print("\nSample Chunk:")
print(chunks[0]["text"])

Text chunking completed!
Total chunks created: 649

Sample Chunk:
SCHOOL OF ENGINEERING AND TECHNOLOGY Programme Handbook (Programme Study and Evaluation Scheme) B.Tech (Computer Science & Engineering) with Specialization in AI & ML Programme Code: 73 FOUR YEAR UNDERGRADUATE PROGRAMME As per National Education Policy 2020 (Multiple Entry and Exit in Academic Programmes) (with effect from 2024 -25 session) Approved in the 34th Meeting of Academic Council Held on 29 June 2024


In [7]:
# Generate Embeddings Using Sentence Transformers

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    batch_size=50,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.array(embeddings, dtype="float32")

print("Embeddings generated successfully!")
print("Embedding Shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Embeddings generated successfully!
Embedding Shape: (649, 384)


In [8]:
dimension = embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(embeddings)

print("FAISS Vector Database created successfully!")
print("Total vectors stored:", faiss_index.ntotal)
print("Vector dimensions:", dimension)

FAISS Vector Database created successfully!
Total vectors stored: 649
Vector dimensions: 384


In [9]:
# Create Semantic Search Function

def search_documents(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    results = []

    for idx, score in zip(indices[0], scores[0]):
        if idx != -1:
            results.append({
                "page": chunks[idx]["page"],
                "text": chunks[idx]["text"],
                "score": float(score)
            })

    return results


# Test Semantic Search

query = "What are the eligibility criteria for admission?"

search_results = search_documents(query)

for result in search_results:
    print("Page:", result["page"])
    print("Score:", round(result["score"], 3))
    print("Text:", result["text"][:300])
    print("-" * 50)

Page: 12
Score: 0.444
Text: explore emerging technologies, and contribute to innovation in academia, industry labs, or R&D departments. 8.6 Duration 4 Years (8 Semesters) - Full-Time Program 8.7 Eligibility Criteria Passed 10+2 examination with Physics and Mathematics as mandatory course. For remaining single course select any
--------------------------------------------------
Page: 208
Score: 0.428
Text: 208 | P a g e Summer Internship-II Program Name: Bachelor of Technology (CSE) with specialization in AI & ML Course Name: Summer Internship-II Course Code L-T-P Credits ENSI351 0-0-0 2 Type of Course: INT Pre-requisite(s), if any: NA Duration: The internship will last for six weeks. It will take pla
--------------------------------------------------
Page: 339
Score: 0.425
Text: 339 | P a g e valuation Criteria Maximum Marks - Content Delivery 8 Marks - Visual Aids and Communication Skills 7 Marks | Total | 100 Marks | Detailed View: 1. Relevance to Learning Outcomes (30 Marks) o Case 

In [10]:
# Connect to Local LLM using Ollama

OLLAMA_URL = "http://localhost:11434/api/generate"

MODEL_NAME = "llama3.2:1b"

response = requests.post(
    OLLAMA_URL,
    json={
        "model": MODEL_NAME,
        "prompt": "Explain what NLP means in one sentence.",
        "stream": False
    },
    timeout=180
)

if response.status_code == 200:
    print("Ollama connected successfully!")
    print("\nAI Response:")
    print(response.json()["response"])
else:
    print("Ollama connection failed:", response.status_code)

Ollama connected successfully!

AI Response:
NLP stands for Natural Language Processing, which is a branch of artificial intelligence that enables computers to understand, interpret, and generate human language, allowing them to perform tasks such as text analysis, language translation, and conversation.


In [11]:
# Connect to Local LLM using Ollama

OLLAMA_URL = "http://localhost:11434/api/generate"

MODEL_NAME = "llama3.2:1b"

response = requests.post(
    OLLAMA_URL,
    json={
        "model": MODEL_NAME,
        "prompt": "Explain what NLP means in one sentence.",
        "stream": False
    },
    timeout=180
)

if response.status_code == 200:
    print("Ollama connected successfully!")
    print("\nAI Response:")
    print(response.json()["response"])
else:
    print("Ollama connection failed:", response.status_code)

Ollama connected successfully!

AI Response:
Natural Language Processing (NLP) is the subfield of artificial intelligence that deals with the interaction between computers and humans in natural language, enabling computers to understand, interpret, and generate human language.


In [12]:
# Generate Answers Using RAG

def generate_answer(question):

    results = search_documents(question, top_k=5)

    context = ""

    for result in results:
        context += (
            f"\nPage {result['page']}:\n"
            f"{result['text']}\n"
        )

    prompt = f"""
You are an AI university document assistant.

Answer the question using ONLY the document context.
Give a clear and simple answer.
Mention the relevant page numbers.
Do not invent information.
If the answer is not available, say so.

DOCUMENT CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 300
            }
        },
        timeout=180
    )

    response.raise_for_status()

    return response.json()["response"], results


print("RAG answer generation function created successfully!")

RAG answer generation function created successfully!


In [13]:
# Test RAG-Based Document Assistant

question = "What are the eligibility criteria for B.Tech CSE AI and ML?"

answer, sources = generate_answer(question)

print("QUESTION:")
print(question)

print("\nAI ANSWER:")
print(answer)

print("\nSOURCE DOCUMENTS:")

for source in sources:
    print(f"\nPage {source['page']}:")
    print(source["text"])
    print("-" * 50)

QUESTION:
What are the eligibility criteria for B.Tech CSE AI and ML?

AI ANSWER:
Page 14 of the document states that the total credit requirement for the B.Tech CSE program is 168 credits. However, it does not provide the eligibility criteria for B.Tech CSE AI and ML.

Page 12 of the document mentions that graduates of the Bachelor of Technology (CSE) with specialization in AI & ML have a wide array of career opportunities, including:

1. Software Developer
2. Systems Analyst
3. Data Scientist
4. AI Engineer
5. Cybersecurity Analyst
6. Network Engineer

However, it does not provide the specific eligibility criteria for these roles.

Therefore, I cannot provide a clear and simple answer to the question.

SOURCE DOCUMENTS:

Page 14:
Structure and Degree Requirements The B.Tech Computer Science & Engineering (CSE) program at the School of Engineering & Technology offers a comprehensive, structured curriculum designed to meet the demands of the rapidly evolving field of computer science. 

In [14]:
# Test RAG-Based Document Assistant

question = "What are the eligibility criteria for B.Tech CSE AI and ML?"

answer, sources = generate_answer(question)

print("QUESTION:")
print(question)

print("\nAI ANSWER:")
print(answer)

print("\nSOURCE DOCUMENTS:")

for source in sources:
    print(f"\nPage {source['page']}:")
    print(source["text"])
    print("-" * 50)

QUESTION:
What are the eligibility criteria for B.Tech CSE AI and ML?

AI ANSWER:
Page 14 of the document states that the total credit requirement for the B.Tech CSE program is 168 credits. However, it does not provide the eligibility criteria for B.Tech CSE AI and ML.

Page 12 of the document mentions that graduates of the Bachelor of Technology (CSE) with specialization in AI & ML have a wide array of career opportunities, including:

1. Software Developer
2. Systems Analyst
3. Data Scientist
4. AI Engineer
5. Cybersecurity Analyst
6. Network Engineer

However, it does not provide the specific eligibility criteria for these roles.

Therefore, I cannot provide a clear and simple answer to the question.

SOURCE DOCUMENTS:

Page 14:
Structure and Degree Requirements The B.Tech Computer Science & Engineering (CSE) program at the School of Engineering & Technology offers a comprehensive, structured curriculum designed to meet the demands of the rapidly evolving field of computer science. 

In [15]:
evaluation_data = {
    "Test Case": [
        "Eligibility Criteria",
        "Total AIML Credits",
        "Total Semesters",
        "Major Credits",
        "Minor Credits",
        "Internship Credits"
    ],
    "Expected Result": [
        "Physics, Mathematics and 50% aggregate",
        "166 Credits",
        "8 Semesters",
        "88 Credits",
        "20 Credits",
        "6 Credits"
    ],
    "Status": [
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Needs Improvement"
    ]
}

evaluation_df = pd.DataFrame(evaluation_data)
display(evaluation_df)

,Test Case,Expected Result,Status
0,Eligibility Criteria,"Physics, Mathematics and 50% aggregate",Pass
1,Total AIML Credits,166 Credits,Pass
2,Total Semesters,8 Semesters,Pass
3,Major Credits,88 Credits,Pass
4,Minor Credits,20 Credits,Pass
5,Internship Credits,6 Credits,Needs Improvement


In [16]:
evaluation_df.to_csv("evaluation_results.csv", index=False)

print("Evaluation results saved successfully!")

Evaluation results saved successfully!


In [17]:
requirements = """
streamlit
pypdf
sentence-transformers
langchain-text-splitters
faiss-cpu
numpy
pandas
requests
"""

with open("requirements.txt", "w") as file:
    file.write(requirements.strip())

print("requirements.txt created successfully!")

requirements.txt created successfully!


In [18]:
readme = """
# RAG-Based Document Assistant

## Project Overview
An NLP-based application that answers questions from a university programme handbook using Retrieval-Augmented Generation (RAG).

## Technologies Used
- Python
- Streamlit
- Sentence Transformers
- FAISS
- Ollama (Llama 3.2)
- PyPDF
- NLP

## Project Workflow
1. Upload the university handbook PDF.
2. Extract text from PDF pages.
3. Clean and split text into chunks.
4. Generate text embeddings.
5. Store embeddings in FAISS.
6. Retrieve relevant chunks using semantic search.
7. Generate answers using the Ollama language model.
8. Display answers with source page references.

## How to Run
1. Install the required libraries:
   pip install -r requirements.txt

2. Start Ollama and ensure llama3.2:1b is available.

3. Run the application:
   streamlit run app.py

## Dataset
K.R. Mangalam University B.Tech CSE (AI & ML) Programme Handbook.

## Output
The application provides answers to user questions along with relevant source text and page numbers.
"""

with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme)

print("README.md created successfully!")

README.md created successfully!
